In [ ]:
# ========================================================================
# SETUP DE PATHS - USAR source/config.py
# ========================================================================

import sys
from pathlib import Path

notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    sys.path.insert(0, str(notebook_dir.parent))
else:
    sys.path.insert(0, str(notebook_dir))

from source.config import (
    PROJ_ROOT, RAW_DATA_DIR, PROCESSED_DATA_DIR, EXTERNAL_DATA_DIR,
    get_data_path
)

from loguru import logger

logger.info(f"📁 Projeto: {PROJ_ROOT}")

# ========================================================================

# indice

- [1 Load libs](#1-Load-libs)
- [2 Config path](#2-Config-path)


## 1 Load libs

In [ ]:
import pandas as pd

# Configuração para exibir todas as colunas
pd.set_option('display.max_columns', None)

# Configuração para exibir todas as linhas
pd.set_option('display.max_rows', None)

# Configuração para que o conteúdo de uma coluna não seja cortado
pd.set_option('display.max_colwidth', None)

# Configuração para expandir a largura da exibição para que mais colunas caibam na tela
pd.set_option('display.width', 1000)

https://gemini.google.com/app

## 2 Config path

In [ ]:
# Caminhos dinâmicos usando config.py (não hardcoded)
path_accounts_df = get_data_path("LI-Medium_accounts.csv", "external")
path_trans_df = get_data_path("LI-Medium_Trans.csv", "external")

logger.info(f"Accounts: {path_accounts_df}")
logger.info(f"Trans: {path_trans_df}")

## 3 Union datasets

In [4]:


# Carregar os arquivos
accounts_df = pd.read_csv(path_accounts_df)
trans_df = pd.read_csv(path_trans_df)

# Renomear colunas duplicadas em trans_df para maior clareza
trans_df.columns = ['Timestamp', 'From Bank', 'From Account', 'To Bank', 'To Account', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']

# 1. Juntar transações com informações da conta de origem (remetente)
trans_enriched_df = pd.merge(
    trans_df,
    accounts_df,
    left_on=['From Bank', 'From Account'],
    right_on=['Bank ID', 'Account Number'],
    how='left'
)

# Renomear colunas para evitar conflitos
trans_enriched_df = trans_enriched_df.rename(columns={
    'Bank Name': 'From Bank Name',
    'Entity ID': 'From Entity ID',
    'Entity Name': 'From Entity Name'
})

# 2. Juntar o resultado com informações da conta de destino (destinatário)
trans_enriched_df = pd.merge(
    trans_enriched_df,
    accounts_df,
    left_on=['To Bank', 'To Account'],
    right_on=['Bank ID', 'Account Number'],
    how='left',
    suffixes=('', '_To')
)

# Renomear colunas para maior clareza
trans_enriched_df = trans_enriched_df.rename(columns={
    'Bank Name': 'To Bank Name',
    'Entity ID': 'To Entity ID',
    'Entity Name': 'To Entity Name'
})

# Exibir o resultado
print("Tabela de Transações Enriquecida:")
print(trans_enriched_df.head())


Tabela de Transações Enriquecida:
          Timestamp  From Bank From Account  To Bank To Account  Amount Received Receiving Currency  Amount Paid Payment Currency Payment Format  Is Laundering           From Bank Name  Bank ID Account Number From Entity ID        From Entity Name             To Bank Name  Bank ID_To Account Number_To To Entity ID          To Entity Name
0  2022/09/01 00:15         20    800104D70       20  800104D70          8095.07          US Dollar      8095.07        US Dollar   Reinvestment              0     Regents Credit Union       20      800104D70    2AA23697070  Sole Proprietorship #1     Regents Credit Union          20         800104D70  2AA23697070  Sole Proprietorship #1
1  2022/09/01 00:18       3196    800107150     3196  800107150          7739.29          US Dollar      7739.29        US Dollar   Reinvestment              0     Bank of Philadelphia     3196      800107150    2AA23466990          Partnership #1     Bank of Philadelphia        3196  

## 3 Salve data raw

In [ ]:
# Salvar usando config.py (não hardcoded path)
output_path = get_data_path('trans_enriched.csv', 'raw')
trans_enriched_df.to_csv(output_path, index=False)

logger.success(f"✅ Dados salvos em: {output_path}")

In [ ]:
trans_enriched_df.head()

,Timestamp,From Bank,From Account,To Bank,To Account,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,From Bank Name,Bank ID,Account Number,From Entity ID,From Entity Name,To Bank Name,Bank ID_To,Account Number_To,To Entity ID,To Entity Name
0,2022/09/01 00:15,20,800104D70,20,800104D70,8095.07,US Dollar,8095.07,US Dollar,Reinvestment,0,Regents Credit Union,20,800104D70,2AA23697070,Sole Proprietorship #1,Regents Credit Union,20,800104D70,2AA23697070,Sole Proprietorship #1
1,2022/09/01 00:18,3196,800107150,3196,800107150,7739.29,US Dollar,7739.29,US Dollar,Reinvestment,0,Bank of Philadelphia,3196,800107150,2AA23466990,Partnership #1,Bank of Philadelphia,3196,800107150,2AA23466990,Partnership #1
2,2022/09/01 00:23,1208,80010E430,1208,80010E430,2654.22,US Dollar,2654.22,US Dollar,Reinvestment,0,Bank of Danbury,1208,80010E430,2AA237140E0,Sole Proprietorship #2,Bank of Danbury,1208,80010E430,2AA237140E0,Sole Proprietorship #2
3,2022/09/01 00:19,3203,80010EA80,3203,80010EA80,13284.41,US Dollar,13284.41,US Dollar,Reinvestment,0,Savings Bank of Augusta,3203,80010EA80,2AA23134470,Sole Proprietorship #3,Savings Bank of Augusta,3203,80010EA80,2AA23134470,Sole Proprietorship #3
4,2022/09/01 00:27,20,800104D20,20,800104D20,9.72,US Dollar,9.72,US Dollar,Reinvestment,0,Regents Credit Union,20,800104D20,2AA1FB24CC0,Individual #1,Regents Credit Union,20,800104D20,2AA1FB24CC0,Individual #1
